# Amazon Beauty 2018: ETL, Cleaning, and Model Preparation

This notebook uses only the Amazon Beauty dataset. It prepares one common model-ready table for:

1. Centralized SASRec
2. FedSASRec with secure aggregation
3. Compressed/on-device SASRec
4. Differentially private SASRec

No Movies and TV data is used in this notebook.


## Dataset

Dataset: Amazon Review Data 2018 — All Beauty category.

Official source: https://cseweb.ucsd.edu/~jmcauley/datasets/amazon_v2/

The review timestamps cover May 1996 through October 2018. The ratings-only file is used for the common sequential-recommendation input. Review text and product metadata are loaded separately as optional information.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)

DATA_DIR = Path("amazon_beauty_data")
OUTPUT_DIR = Path("amazon_beauty_prepared_outputs")
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

# Set True only if the complete review JSON is required.
# The complete file can require substantial memory.
LOAD_METADATA = True
LOAD_FULL_REVIEW_JSON = False
REVIEW_SAMPLE_ROWS = 10_000


In [2]:
RATINGS_URL = (
    "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_v2/"
    "categoryFilesSmall/All_Beauty.csv"
)
REVIEWS_URL = (
    "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_v2/"
    "categoryFiles/All_Beauty.json.gz"
)
METADATA_URL = (
    "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_v2/"
    "metaFiles2/meta_All_Beauty.json.gz"
)

RATINGS_FILE = DATA_DIR / "All_Beauty.csv"
REVIEWS_FILE = DATA_DIR / "All_Beauty.json.gz"
METADATA_FILE = DATA_DIR / "meta_All_Beauty.json.gz"

print("Officially reported interaction/review records: 371,345")
print("Officially reported product metadata records: 32,992")


Officially reported interaction/review records: 371,345
Officially reported product metadata records: 32,992


## Extract: load the Amazon Beauty ratings-only data

The raw ratings-only file has four core columns: user, item, rating, and timestamp. The loader supports different Amazon header names and the official item-user-rating-timestamp ordering.


In [3]:
def load_ratings_csv():
    source = RATINGS_FILE if RATINGS_FILE.exists() else RATINGS_URL
    raw = pd.read_csv(source)

    aliases = {
        "userid": "user_id",
        "user_id": "user_id",
        "reviewerid": "user_id",
        "user": "user_id",
        "productid": "item_id",
        "itemid": "item_id",
        "item_id": "item_id",
        "asin": "item_id",
        "item": "item_id",
        "rating": "rating",
        "overall": "rating",
        "timestamp": "timestamp",
        "unixreviewtime": "timestamp",
    }

    normalized = {
        column: str(column).strip().lower().replace(" ", "_")
        for column in raw.columns
    }
    renamed = {
        column: aliases.get(normalized_name, normalized_name)
        for column, normalized_name in normalized.items()
    }
    raw = raw.rename(columns=renamed)

    required = {"user_id", "item_id", "rating", "timestamp"}
    if not required.issubset(raw.columns):
        # Official ratings-only files may be ordered item, user, rating, timestamp.
        raw = pd.read_csv(
            source,
            header=None,
            names=["item_id", "user_id", "rating", "timestamp"],
        )

    return raw[["user_id", "item_id", "rating", "timestamp"]].copy()

raw = load_ratings_csv()
print("Raw rows:", f"{len(raw):,}")
print("Raw columns:", len(raw.columns))
print("Raw column names:", list(raw.columns))
raw.head()


Raw rows: 371,345
Raw columns: 4
Raw column names: ['user_id', 'item_id', 'rating', 'timestamp']


,user_id,item_id,rating,timestamp
0,A1V6B6TNIC10QE,0143026860,1.0,1424304000
1,A2F5GHSXFQ0W6J,0143026860,4.0,1418860800
2,A1572GUYS7DGSR,0143026860,4.0,1407628800
3,A1PSGLFK1NSVO,0143026860,5.0,1362960000
4,A6IKXKZMTKGSC,0143026860,5.0,1324771200


## Transform: clean and filter interactions

Cleaning operations:

- Remove null and empty user/item IDs.
- Convert rating and timestamp to numeric values.
- Remove ratings outside the 1–5 range.
- Remove invalid timestamps.
- Remove exact duplicate rows.
- Keep one chronological record per repeated user-item pair.
- Convert ratings 4 and 5 into positive implicit feedback.
- Keep users with at least five positive interactions.


In [4]:
def clean_beauty_interactions(
    data,
    positive_threshold=4,
    min_interactions=5,
):
    cleaned = data.copy()
    before_rows = len(cleaned)

    cleaned["user_id"] = cleaned["user_id"].astype("string").str.strip()
    cleaned["item_id"] = cleaned["item_id"].astype("string").str.strip()
    cleaned["rating"] = pd.to_numeric(
        cleaned["rating"],
        errors="coerce",
    )
    cleaned["timestamp"] = pd.to_numeric(
        cleaned["timestamp"],
        errors="coerce",
    )

    cleaned = cleaned.dropna(
        subset=["user_id", "item_id", "rating", "timestamp"]
    )
    cleaned = cleaned[
        cleaned["user_id"].ne("")
        & cleaned["item_id"].ne("")
    ]
    cleaned = cleaned[cleaned["rating"].between(1, 5)]
    cleaned = cleaned[cleaned["timestamp"] > 0]

    cleaned = cleaned.sort_values(
        ["user_id", "timestamp", "item_id"]
    )
    cleaned = cleaned.drop_duplicates()
    cleaned = cleaned.drop_duplicates(
        subset=["user_id", "item_id"],
        keep="first",
    )

    cleaned = cleaned[
        cleaned["rating"] >= positive_threshold
    ].copy()

    user_counts = cleaned.groupby("user_id").size()
    eligible_users = user_counts[
        user_counts >= min_interactions
    ].index
    cleaned = cleaned[
        cleaned["user_id"].isin(eligible_users)
    ].copy()

    cleaned["domain"] = "beauty"
    cleaned["datetime"] = pd.to_datetime(
        cleaned["timestamp"],
        unit="s",
        errors="coerce",
    )
    cleaned = cleaned.sort_values(
        ["user_id", "timestamp", "item_id"]
    ).reset_index(drop=True)

    print(f"Rows before cleaning: {before_rows:,}")
    print(f"Rows after cleaning:  {len(cleaned):,}")
    print(f"Rows removed:         {before_rows - len(cleaned):,}")
    print(f"Users retained:       {cleaned['user_id'].nunique():,}")
    print(f"Items retained:       {cleaned['item_id'].nunique():,}")
    return cleaned

cleaned = clean_beauty_interactions(raw)
cleaned.head()


Rows before cleaning: 371,345
Rows after cleaning:  3,313
Rows removed:         368,032
Users retained:       590
Items retained:       1,211


,user_id,item_id,rating,timestamp,domain,datetime
0,A105A034ZG9EHO,B000CR4ER6,4.0,1155081600,beauty,2006-08-09
1,A105A034ZG9EHO,B000EIOAFY,5.0,1268697600,beauty,2010-03-16
2,A105A034ZG9EHO,B0009RF9DW,5.0,1404604800,beauty,2014-07-06
3,A105A034ZG9EHO,B000FI4S1E,5.0,1404604800,beauty,2014-07-06
4,A105A034ZG9EHO,B000URXP6E,5.0,1404604800,beauty,2014-07-06


## Optional data: reviews and product metadata

Review fields and product metadata are retained separately. They are not required for the common SASRec input, but they can support content features, cold-start analysis, and recommendation explanations.


In [5]:
def load_json_lines(path, url, nrows=None):
    source = path if path.exists() else url
    return pd.read_json(
        source,
        lines=True,
        compression="gzip",
        nrows=nrows,
    )

if LOAD_METADATA:
    metadata = load_json_lines(
        METADATA_FILE,
        METADATA_URL,
    )
    metadata = metadata.rename(columns={"asin": "item_id"})
    metadata["item_id"] = (
        metadata["item_id"].astype("string").str.strip()
    )
    metadata = metadata.drop_duplicates(
        subset=["item_id"]
    )
    metadata["domain"] = "beauty"
    print("Metadata rows loaded:", f"{len(metadata):,}")
    print("Metadata columns:", len(metadata.columns))
else:
    metadata = pd.DataFrame()
    print("Metadata loading disabled.")

review_rows = None if LOAD_FULL_REVIEW_JSON else REVIEW_SAMPLE_ROWS
reviews = load_json_lines(
    REVIEWS_FILE,
    REVIEWS_URL,
    nrows=review_rows,
)
print("Review rows loaded:", f"{len(reviews):,}")
print("Review columns:", len(reviews.columns))
reviews.head()


Metadata rows loaded: 32,488
Metadata columns: 20
Review rows loaded: 10,000
Review columns: 12


,overall,verified,reviewTime,reviewerID,asin,reviewerName,reviewText,summary,unixReviewTime,vote,style,image
0,1,True,"02 19, 2015",A1V6B6TNIC10QE,0143026860,theodore j bigham,great,One Star,1424304000,NaN,NaN,NaN
1,4,True,"12 18, 2014",A2F5GHSXFQ0W6J,0143026860,Mary K. Byke,My husband wanted to reading about the Negro ...,... to reading about the Negro Baseball and th...,1418860800,NaN,NaN,NaN
2,4,True,"08 10, 2014",A1572GUYS7DGSR,0143026860,David G,"This book was very informative, covering all a...",Worth the Read,1407628800,NaN,NaN,NaN
3,5,True,"03 11, 2013",A1PSGLFK1NSVO,0143026860,TamB,I am already a baseball fan and knew a bit abo...,Good Read,1362960000,NaN,NaN,NaN
4,5,True,"12 25, 2011",A6IKXKZMTKGSC,0143026860,shoecanary,This was a good story of the Black leagues. I ...,"More than facts, a good story read!",1324771200,5,NaN,NaN


## Transform: create the common 12-column model-ready table

Every model receives the same prepared interaction columns. The differences between the models come from training and deployment, not from using different raw data.


In [6]:
all_interactions = cleaned.copy()

all_interactions["global_item_key"] = (
    "beauty:" + all_interactions["item_id"]
)
all_interactions["client_id"] = (
    "beauty:" + all_interactions["user_id"]
)

user_values = sorted(all_interactions["user_id"].unique())
item_values = sorted(
    all_interactions["global_item_key"].unique()
)

user_to_index = {
    value: index
    for index, value in enumerate(user_values, start=1)
}
item_to_index = {
    value: index
    for index, value in enumerate(item_values, start=1)
}

all_interactions["user_index"] = (
    all_interactions["user_id"]
    .map(user_to_index)
    .astype("int64")
)
all_interactions["item_index"] = (
    all_interactions["global_item_key"]
    .map(item_to_index)
    .astype("int64")
)

all_interactions = all_interactions.sort_values(
    ["user_id", "timestamp", "item_id"]
).reset_index(drop=True)

print("Columns before train/validation/test split:")
print(len(all_interactions.columns))
print(list(all_interactions.columns))


Columns before train/validation/test split:
10
['user_id', 'item_id', 'rating', 'timestamp', 'domain', 'datetime', 'global_item_key', 'client_id', 'user_index', 'item_index']


In [7]:
def chronological_split(user_group):
    user_group = user_group.sort_values(
        ["timestamp", "item_id"]
    ).copy()

    labels = ["train"] * len(user_group)
    labels[-2] = "validation"
    labels[-1] = "test"

    user_group["split"] = labels
    user_group["sequence_position"] = range(
        1,
        len(user_group) + 1,
    )
    return user_group

split_frames = []

for _, user_group in all_interactions.groupby(
    "user_id",
    sort=False,
):
    split_frames.append(
        chronological_split(user_group)
    )

prepared = pd.concat(
    split_frames,
    ignore_index=True,
)
prepared = prepared.sort_values(
    ["user_index", "sequence_position"]
).reset_index(drop=True)

print(prepared["split"].value_counts())
print("Prepared columns:", len(prepared.columns))
print(list(prepared.columns))


split
train         2133
validation     590
test           590
Name: count, dtype: int64
Prepared columns: 12
['user_id', 'item_id', 'rating', 'timestamp', 'domain', 'datetime', 'global_item_key', 'client_id', 'user_index', 'item_index', 'split', 'sequence_position']


In [8]:
# Verify chronology and required values.
for _, user_group in prepared.groupby("user_id"):
    train_time = user_group.loc[
        user_group["split"] == "train",
        "timestamp",
    ].max()
    validation_time = user_group.loc[
        user_group["split"] == "validation",
        "timestamp",
    ].min()
    test_time = user_group.loc[
        user_group["split"] == "test",
        "timestamp",
    ].min()

    assert train_time <= validation_time <= test_time

required_columns = [
    "user_id",
    "item_id",
    "rating",
    "timestamp",
    "datetime",
    "domain",
    "global_item_key",
    "client_id",
    "user_index",
    "item_index",
    "split",
    "sequence_position",
]

missing_columns = [
    column for column in required_columns
    if column not in prepared.columns
]

assert not missing_columns, missing_columns
assert not prepared[required_columns].isna().any().any()

model_ready = prepared[required_columns].copy()

print("Temporal leakage check passed.")
print("Final model-ready column count:", len(model_ready.columns))
print("Final model-ready columns:")
print(list(model_ready.columns))


Temporal leakage check passed.
Final model-ready column count: 12
Final model-ready columns:
['user_id', 'item_id', 'rating', 'timestamp', 'datetime', 'domain', 'global_item_key', 'client_id', 'user_index', 'item_index', 'split', 'sequence_position']


## Column mapping for the four models

- Centralized SASRec uses user_id, item_id, timestamp, item_index, sequence_position, and split.
- FedSASRec uses the same fields plus client_id and user_index.
- Compressed/on-device SASRec uses the same interaction table; model size, memory, latency, and local update time are measured separately.
- Differentially private SASRec uses the same interaction table; epsilon, delta, gradient clipping, and noise are training settings.
- Secure aggregation is a training protocol, not a dataset column.
- Review text and product metadata are optional extensions.


In [9]:
# Save all model-ready and optional ETL outputs.
model_ready.to_csv(
    OUTPUT_DIR / "amazon_beauty_model_ready.csv",
    index=False,
)
model_ready[model_ready["split"] == "train"].to_csv(
    OUTPUT_DIR / "amazon_beauty_train.csv",
    index=False,
)
model_ready[
    model_ready["split"] == "validation"
].to_csv(
    OUTPUT_DIR / "amazon_beauty_validation.csv",
    index=False,
)
model_ready[model_ready["split"] == "test"].to_csv(
    OUTPUT_DIR / "amazon_beauty_test.csv",
    index=False,
)

client_sequences = (
    model_ready.sort_values(
        ["client_id", "sequence_position"]
    )
    .groupby(
        ["client_id", "user_id", "user_index"]
    )["item_index"]
    .apply(lambda values: " ".join(map(str, values)))
    .reset_index(name="item_sequence")
)
client_sequences.to_csv(
    OUTPUT_DIR / "amazon_beauty_client_sequences.csv",
    index=False,
)

if LOAD_METADATA:
    metadata.to_csv(
        OUTPUT_DIR / "amazon_beauty_metadata.csv",
        index=False,
    )

reviews.to_json(
    OUTPUT_DIR / "amazon_beauty_reviews_loaded.jsonl",
    orient="records",
    lines=True,
)

print(f"Saved outputs in: {OUTPUT_DIR.resolve()}")
for file_path in sorted(OUTPUT_DIR.glob("*")):
    print("-", file_path.name)


Saved outputs in: /content/amazon_beauty_prepared_outputs
- amazon_beauty_client_sequences.csv
- amazon_beauty_metadata.csv
- amazon_beauty_model_ready.csv
- amazon_beauty_reviews_loaded.jsonl
- amazon_beauty_test.csv
- amazon_beauty_train.csv
- amazon_beauty_validation.csv


In [10]:
train_check = pd.read_csv(
    OUTPUT_DIR / "amazon_beauty_train.csv"
)
validation_check = pd.read_csv(
    OUTPUT_DIR / "amazon_beauty_validation.csv"
)
test_check = pd.read_csv(
    OUTPUT_DIR / "amazon_beauty_test.csv"
)

assert train_check["split"].eq("train").all()
assert validation_check["split"].eq("validation").all()
assert test_check["split"].eq("test").all()

assert (
    len(train_check)
    + len(validation_check)
    + len(test_check)
    == len(model_ready)
)

print("Reload verification passed.")
print(f"Train rows:      {len(train_check):,}")
print(f"Validation rows: {len(validation_check):,}")
print(f"Test rows:       {len(test_check):,}")
print(f"Final columns:   {len(train_check.columns)}")


Reload verification passed.
Train rows:      2,133
Validation rows: 590
Test rows:       590
Final columns:   12


## Final result

The common file amazon_beauty_model_ready.csv contains exactly 12 columns and is the shared input for all four models.

Only Amazon Beauty data is used. No Movies and TV data is loaded or merged.
